# Setup

Change `/workspaces/taegis-sdk-python-workspace/taegis-magic` to wherever your taegis-magic dir is that is on the correct branch.<br><br>
After changing the directory make sure to reload the kernel!

In [ ]:
!pip install -e /workspaces/taegis-sdk-python-workspace/taegis-magic

Installing itables for the purposes of displaying DataFrames in a nicer format

In [ ]:
!pip install itables

In [ ]:
from taegis_magic.pandas.process import process_correlate_netflow
import pandas as pd
from itables import show


In [ ]:
%load_ext taegis_magic

In [ ]:
%taegis users current-user --assign me --display me

The fully query below can be accessed via: https://ctpx.secureworks.com/share/c88dab66-7876-44c8-998f-a7ca97851e8f

The query specifies two process_correlation_ids where each one correlates to a different netflow event. One netflow event has a `processcorrelationid.pid` with a format of `pid:timewindow` with an empty `processcorrelationid.timewindow`, and the other a `processcorrelationid.pid` with a format of `pid` with a non empty `processcorrelationid.timewindow` value.

The query is just to get some valid process data to put into the new function, `process_correlate_netflow`

In [ ]:
%%taegis events search --assign process_events
FROM process 
WHERE 
    process_correlation_id in ('process_correlation_id_1', 'process_correlation_id_2')
EARLIEST=-30d

# Beginning of Testing

Edit the `process_events` variable returned by the above query to only contain the process_correlation_id column for the sake of simplicity/readability. 

In [ ]:
process_events = process_events[["process_correlation_id"]]

In [ ]:
print(process_events)

Setting log level to DEBUG to show more detail if desired

In [ ]:
# import logging
# logging.basicConfig(level=logging.DEBUG)
# logging.getLogger("taegis_magic").setLevel(logging.DEBUG)

## Testing Valid Data

Now calling new function, `process_correlate_netflow` with valid data

In [ ]:
ret_valid_data = process_correlate_netflow(process_events, "charlie")

In [ ]:
print(len(ret_valid_data))

Note, that while there were originally only two rows in the input DataFrame that more rows may be returned due to the 1:many relationship between process events and netflow events. 

In [ ]:
show(ret_valid_data)

## Testing putting DataFrame returned by `process_correlate_netflow` as input to the function
In this case, it should just return the DataFrame that was put into the function

In [ ]:
process_correlate_netflow(ret_valid_data, "charlie")

## Testing providing a different value for `process_column`

First, will create a copy of the `process_events` from earlier and change the name

In [ ]:
new_process_column = "my_pid"
process_events_diff_col = process_events.copy()
process_events_diff_col.rename(columns={"process_correlation_id":new_process_column},inplace=True)

In [ ]:
print(process_events_diff_col)

In [ ]:
ret_with_diff_process_column = process_correlate_netflow(process_events_diff_col, "charlie", None, new_process_column)

In [ ]:
show(ret_with_diff_process_column)

## Testing where `process_column` is not in DataFrame

In [ ]:
ret_valid_data = process_correlate_netflow(process_events, "charlie", None, new_process_column)